# Demand Forecast + Sentiment + Explainability

This notebook loads Olist data from S3, builds a product-level demand forecast, runs sentiment analysis with Amazon Comprehend, and generates business explanations with Amazon Bedrock.

## Setup

In [ ]:
import pandas as pd
import boto3
import json
from io import StringIO

In [ ]:
bucket = "bluepill-ai-data"
prefix = "raw/"

s3 = boto3.client('s3')

def read_csv_from_s3(key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(obj['Body'])

## Load and Prepare Demand Data

In [ ]:
orders = read_csv_from_s3("raw/olist_orders_dataset.csv")
items = read_csv_from_s3("raw/olist_order_items_dataset.csv")

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['date'] = orders['order_purchase_timestamp'].dt.date

merged = items.merge(orders[['order_id', 'date']], on='order_id', how='left')
forecast_data = merged.groupby(['date', 'product_id']).size().reset_index(name='sales')
forecast_data.head()

In [ ]:
forecast_data['date'] = pd.to_datetime(forecast_data['date'])
forecast_data = forecast_data.sort_values(by=['product_id', 'date'])
forecast_data.head()

## Feature Engineering

In [ ]:
top_product = forecast_data['product_id'].value_counts().idxmax()
product_data = forecast_data[forecast_data['product_id'] == top_product].copy()
product_data.head()

In [ ]:
product_data['day'] = product_data['date'].dt.day
product_data['month'] = product_data['date'].dt.month
product_data['weekday'] = product_data['date'].dt.weekday
product_data.head()

## Train/Test Split

In [ ]:
train = product_data.iloc[:-30]
test = product_data.iloc[-30:]

X_train = train[['day', 'month', 'weekday']]
y_train = train['sales']

X_test = test[['day', 'month', 'weekday']]
y_test = test['sales']

## Train Forecast Model

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

model = xgb.XGBRegressor(
    n_estimators=50,
    max_depth=3,
    learning_rate=0.1
)

model.fit(X_train, y_train)
preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
print("MAE:", mae)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.plot(test['date'], y_test, label="Actual")
plt.plot(test['date'], preds, label="Predicted")
plt.legend()
plt.title("Demand Forecast (BluePill AI)")
plt.show()

## Sentiment Analysis with Amazon Comprehend

In [ ]:
comprehend = boto3.client('comprehend')

reviews = read_csv_from_s3("raw/olist_order_reviews_dataset.csv")
reviews = reviews[['review_comment_message']].dropna()
reviews = reviews.head(200)
reviews.head()

In [ ]:
sentiments = []

for text in reviews['review_comment_message']:
    try:
        response = comprehend.detect_sentiment(
            Text=text[:500],
            LanguageCode="en"
        )
        sentiments.append(response['Sentiment'])
    except:
        sentiments.append("UNKNOWN")

reviews['sentiment'] = sentiments
reviews.head()

In [ ]:
sentiment_summary = reviews['sentiment'].value_counts()
sentiment_summary

In [ ]:
sentiment_summary.plot(kind='bar')
plt.title("Customer Sentiment Distribution")
plt.show()

In [ ]:
csv_buffer = StringIO()
reviews.to_csv(csv_buffer, index=False)

s3.put_object(Bucket=bucket, Key="processed/sentiment_results.csv", Body=csv_buffer.getvalue())

## Trend Ratio + Recommendations

In [ ]:
avg_predicted = preds.mean()
avg_actual = y_test.mean()

trend_ratio = avg_predicted / avg_actual
trend_ratio

In [ ]:
positive_ratio = sentiment_summary.get("POSITIVE", 0) / len(reviews)
negative_ratio = sentiment_summary.get("NEGATIVE", 0) / len(reviews)

positive_ratio, negative_ratio

In [ ]:
recommendations = []

if trend_ratio > 1.05:
    recommendations.append("Demand is increasing. Consider increasing inventory allocation.")
elif trend_ratio < 0.95:
    recommendations.append("Demand is declining. Optimize inventory to reduce holding costs.")
else:
    recommendations.append("Demand is stable. Maintain current inventory strategy.")

if negative_ratio > 0.3:
    recommendations.append("High negative sentiment detected. Investigate delivery or product issues.")
elif positive_ratio > 0.6:
    recommendations.append("Strong positive sentiment. Opportunity to increase marketing investment.")

recommendations

## Explainability with Amazon Bedrock

In [ ]:
bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1"
)

prompt = f"""
You are an AI retail business consultant.

Forecast trend ratio: {trend_ratio}
Positive sentiment ratio: {positive_ratio}
Negative sentiment ratio: {negative_ratio}

Generated recommendations:
{recommendations}

Explain clearly why these recommendations were made in business terms.
"""

body = {
    "prompt": prompt,
    "max_gen_len": 300,
    "temperature": 0.3,
    "top_p": 0.9
}

response = bedrock.invoke_model(
    modelId="meta.llama3-8b-instruct-v1:0",
    body=json.dumps(body),
    contentType="application/json",
    accept="application/json"
)

response_body = json.loads(response["body"].read())

print(response_body["generation"])

## Save Final Output to S3

In [ ]:
final_output = {
    "forecast": {
        "trend_ratio": float(trend_ratio),
        "mae": float(mae)
    },
    "sentiment": {
        "positive_ratio": float(positive_ratio),
        "negative_ratio": float(negative_ratio)
    },
    "recommendations": recommendations,
    "explanation": response_body["generation"]
}

s3.put_object(
    Bucket=bucket,
    Key="processed/final_recommendation.json",
    Body=json.dumps(final_output)
)
final_output